Fish to Fish Regression Test Errors by Species

In [40]:

import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)


df = pd.read_csv(
    "../adore_dataset/processed/t-F2F_mortality.csv",
    dtype=str
)

#species columns 
df["Full_Species"] = (
    df["tax_genus"].astype(str)
    + " "
    + df["tax_species"].astype(str)
)

species_col = "Full_Species"

# features
target = "result_conc1_mean_mol_log"

categorical_cols = [
    "test_exposure_type",
    "result_conc1_type",
    "test_media_type"
]

ordinal_cols = ["result_obs_duration_mean"]

numeric_cols = [
    "chem_mw",
    "chem_ws",
    "chem_mp",
    "chem_rdkit_clogp"
]

# numeric columns
for col in numeric_cols + ordinal_cols + [target]:
    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    )

# macc fingerprint
def parse_maccs(value):
    if not isinstance(value, str):
        return None

    value = value.strip()

    if len(value) != 167:
        return None

    if not set(value).issubset({"0", "1"}):
        return None

    return [int(bit) for bit in value]

valid_fp = df["chem_MACCS_fp"].apply(parse_maccs)

df = df.loc[valid_fp.notna()].copy()

maccs = pd.DataFrame(
    valid_fp.loc[df.index].tolist(),
    index=df.index,
    columns=[f"MACCS_{i:03d}" for i in range(167)]
)




In [41]:
# X and y
X = pd.concat([
    df[categorical_cols + ordinal_cols + numeric_cols],
    maccs
], axis=1)

y = df[target]

# test splits
train = df["split_occurrence"] != "test"
test = df["split_occurrence"] == "test"

valid = y.notna() & df[species_col].notna()

train = train & y.notna()
test = test & valid

X_train = X.loc[train]
X_test = X.loc[test]

y_train = y.loc[train]
y_test = y.loc[test]

In [42]:
# maccs bits
fp_cols = [
    c for c in maccs.columns
    if X_train[c].std() >= 0.1
]

features = (
    categorical_cols
    + ordinal_cols
    + numeric_cols
    + fp_cols
)

X_train = X_train[features]
X_test = X_test[features]

In [43]:
# preprocessing
numeric_features = (
    ordinal_cols + numeric_cols + fp_cols
)

preprocessing = ColumnTransformer([
    (
        "numeric",
        make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ),
        numeric_features
    ),
    (
        "categorical",
        make_pipeline(
            SimpleImputer(
                strategy="most_frequent"
            ),
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
        categorical_cols
    )
])


In [44]:
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor)
from sklearn.base import clone
from sklearn.metrics import (mean_absolute_error,root_mean_squared_error,r2_score)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}



In [45]:
# train and error calculation
model_results = []
all_predictions = []

for name, regressor in models.items():

    pipeline = make_pipeline(
        clone(preprocessing),
        clone(regressor)
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    model_results.append({
        "Model": name,
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

    predictions = pd.DataFrame({
        "Species": df.loc[X_test.index, species_col].values,
        "Actual": y_test.values,
        "Predicted": y_pred,
        "Model": name
    })

    predictions["Absolute_Error"] = abs(
        predictions["Actual"] - predictions["Predicted"]
    )

    all_predictions.append(predictions)

In [48]:
# comparing models
model_comparison = pd.DataFrame(
    model_results
).sort_values("RMSE")

print("\nMODEL COMPARISON")
print(model_comparison.to_string(index=False))

# all predictions concatonaed
all_errors = pd.concat(
    all_predictions, ignore_index=True
)

# error by species and model

species_model_errors = (
    all_errors.groupby(["Model", "Species"])
    .agg(
        MAE= (mean_absolute_error(y_test, y_pred)),
        RMSE=(
            "Absolute_Error",
            lambda x: np.sqrt(np.mean(x**2))
        ),
        Count=("Absolute_Error", "size")
    )
    .reset_index()
)


MODEL COMPARISON
            Model     RMSE      MAE       R2
              SVR 0.986290 0.723425 0.684615
    Random Forest 1.040736 0.775941 0.648833
Gradient Boosting 1.051203 0.827325 0.641734
              KNN 1.111178 0.802604 0.599687
Linear Regression 1.134927 0.917361 0.582393
            Ridge 1.134931 0.917339 0.582390
    Decision Tree 1.342713 0.982221 0.415481
      Elastic Net 1.475269 1.198285 0.294374
            Lasso 1.688802 1.369309 0.075325


TypeError: Must provide 'func' or tuples of '(column, aggfunc).

In [49]:
# top 10 highest errors
for name in models:
    print(f"\n{name} - Highest Error Species")

    top = (
        species_model_errors[
            species_model_errors["Model"] == name
        ]
        .sort_values("MAE", ascending=False)
        .head(10)
    )

    print(top.to_string(index=False))

#compare spceies
species_summary = (
    species_model_errors.groupby("Species")
    .agg(
        Average_MAE=("MAE", "mean"),
        Average_RMSE=("RMSE", "mean"),
        Count=("Count", "first")
    )
    .sort_values("Average_MAE", ascending=False)
)

print("\nSPECIES WITH GREATEST AVERAGE ERROR")
print(species_summary.head(15).to_string())





Linear Regression - Highest Error Species
            Model                Species      MAE     RMSE  Count
Linear Regression           Gadus morhua 2.757201 2.758106      2
Linear Regression        Clupea harengus 2.025726 2.034095      4
Linear Regression       Monopterus albus 1.900071 1.900082      4
Linear Regression            Esox lucius 1.889307 2.086975      9
Linear Regression      Leuresthes tenuis 1.871054 1.883200     26
Linear Regression       Fundulus grandis 1.829416 1.829416      1
Linear Regression Nothobranchius furzeri 1.748734 1.751178      8
Linear Regression     Gambusia holbrooki 1.633214 1.633265      2
Linear Regression     Lagodon rhomboides 1.622679 1.795703      4
Linear Regression          Sparus aurata 1.615090 1.690368      2

Ridge - Highest Error Species
Model                Species      MAE     RMSE  Count
Ridge           Gadus morhua 2.756712 2.757617      2
Ridge        Clupea harengus 2.024641 2.033016      4
Ridge       Monopterus albus 1.900990 

In [50]:
# save errors for report in csv
model_comparison.to_csv(
    "model_comparison.csv", index=False
)

species_model_errors.to_csv(
    "species_model_errors.csv", index=False
)

species_summary.to_csv(
    "species_error_summary.csv"
)


In [55]:
fish_top10_filtered = (
    species_summary[species_summary["Count"] >= 5]
    .sort_values("Average_MAE", ascending=False)
    .head(10)
)

print(fish_top10_filtered)

                             Average_MAE  Average_RMSE  Count
Species                                                      
Leuresthes tenuis               1.957525      1.983137     26
Esox lucius                     1.758818      2.016802      9
Nothobranchius furzeri          1.524964      1.555768      8
Channa punctata                 1.517904      2.096704    100
Menidia menidia                 1.517711      1.700536     42
Carassius carassius             1.347858      1.635202     13
Hypophthalmichthys molitrix     1.281886      1.356019     14
Clarias gariepinus              1.210199      1.544609     17
Morone saxatilis                1.156345      1.395242     38
Tinca tinca                     1.132982      1.296815      5
